# 0. Problem
## 585. Investments in 2016 — Medium
Sum `tiv_2016` for rows whose `tiv_2015` value is shared by multiple policyholders while their `(lat, lon)` location is unique. Round to 2 decimals.
Official: https://leetcode.com/problems/investments-in-2016/

# 1. Setup

In [ ]:
import pandas as pd
insurance_rows=[(1,10.0,5.0,10.0,10.0),(2,20.0,20.0,20.0,20.0),(3,10.0,30.0,20.0,20.0),(4,10.0,40.0,40.0,40.0)]
insurance_pd=pd.DataFrame(insurance_rows,columns=["pid","tiv_2015","tiv_2016","lat","lon"]); insurance_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate(); insurance_spark=spark.createDataFrame(insurance_rows,["pid","tiv_2015","tiv_2016","lat","lon"]); insurance_spark.createOrReplaceTempView("Insurance")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""WITH x AS (SELECT *,COUNT(*) OVER(PARTITION BY tiv_2015) AS tiv_count,COUNT(*) OVER(PARTITION BY lat,lon) AS loc_count FROM Insurance) SELECT ROUND(SUM(tiv_2016),2) AS tiv_2016 FROM x WHERE tiv_count>1 AND loc_count=1"""); sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
work=insurance_pd.copy(); work["tiv_count"]=work.groupby("tiv_2015")["pid"].transform("size"); work["loc_count"]=work.groupby(["lat","lon"])["pid"].transform("size"); result_pd=pd.DataFrame({"tiv_2016":[round(work.loc[work["tiv_count"].gt(1)&work["loc_count"].eq(1),"tiv_2016"].sum(),2)]}); result_pd

# 4. PySpark Solution

In [ ]:
w_tiv=Window.partitionBy("tiv_2015"); w_loc=Window.partitionBy("lat","lon"); work=insurance_spark.withColumn("tiv_count",F.count("*").over(w_tiv)).withColumn("loc_count",F.count("*").over(w_loc)); result_spark=work.filter((F.col("tiv_count")>1)&(F.col("loc_count")==1)).agg(F.round(F.sum("tiv_2016"),2).alias("tiv_2016")); result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| frequency within group | window `COUNT()` | `.groupby().transform("size")` | Window `count()` |
| compound uniqueness | partition by `(lat,lon)` | group by two columns | Window partition by two cols |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Insurance

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: insurance_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: insurance_spark